In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
df = pd.read_csv("analysis_dataset.csv")
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]
df

,hackathon_name,project_name,project_link,is_winner,inspiration,what_it_does,how_we_built_it,challenges,accomplishments,what_we_learned,what_is_next,tech_stack,external_links,github_links,github_summary
0,Cal Hacks 10.0,Audiogroph,https://devpost.com/software/audiogroph,False,With covid ending and people finding new music...,"The project shows events, creators and any NFT...",We used react to build it in vscode and collab...,We had to learn react and navigate through git...,We have successfully built the frontend of the...,Start planning earlier and commit to branches ...,We need a backend to our project.,"css3, html5, javascript, react",['https://github.com/RyanGertz/CalHacksproject'],['https://github.com/RyanGertz/CalHacksproject'],"The GitHub repository, ""CalHacksproject,"" is a..."
1,Cal Hacks 10.0,Let It Out,https://devpost.com/software/let-it-out-b2xp8u,False,Understanding and expressing emotions can be a...,The user is first prompted to record a vocal b...,Let It Out is a full stack web app. The front ...,The main challenges we ran into came in our fi...,NaN,We learned how to integrate modern technologie...,NaN,"chakra, flask, hume, mongodb, node.js, openai,...","['https://github.com/apolyeti/calhacks10.0,', ...",['https://github.com/apolyeti/calhacks10.0'],"This repository, named `calhacks10.0`, serves ..."
2,Cal Hacks 10.0,Mood Theremin,https://devpost.com/software/mood-theremin,False,"The theremin, an electronic musical instrument...",Mood Theremin translates your facial expressio...,Our project integrates facial expression analy...,NaN,NaN,NaN,NaN,"css, flask, html, hume, javascript, json, pyth...",['https://github.com/Sepulchre49/CalHacks23'],['https://github.com/Sepulchre49/CalHacks23'],The Mood-Theremin repository details a CalHack...
3,Cal Hacks 10.0,Bay Area Safer Housing (B.A.S.H.),https://devpost.com/software/bay-area-safer-ho...,False,The inspiration initially stems from one membe...,B.A.S.H. searches for affordable housing optio...,Our website's foundation was built on next.js ...,Our difficulties lay in our inexperience with ...,"Conceptually, we believe that our project is m...",We've learned most importantly how to better a...,We are serious in our project and are intent o...,"catalog.data.gov, data.sfgov.org, google-maps,...","['https://github.com/matthewaol/b.a.s.h.,']",['https://github.com/matthewaol/b.a.s.h.'],b.a.s.h. (Bay Area Safer Housing) is a CalHack...
4,Cal Hacks 10.0,PawPals,https://devpost.com/software/pawpals-bd8njs,False,"As college students, we both agreed that fun t...","Based on a list of criteria, we prompt OpenAI'...","On the front-end, we utilized Figma to develop...",Some challenges we ran into were the lack of e...,"As a team of two, we were very proud of our fi...",We learned the value of coming up with a plan ...,We hope to deploy the site on Vercel and devel...,"fastapi, figma, openai, python, react.js, sqlite",['https://github.com/Yatsz/PawPals'],['https://github.com/Yatsz/PawPals'],This analysis summarizes the GitHub repository...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6347,HackRice 14 2024,Degree Map AI,https://devpost.com/software/degree-planner-ai,False,As current undergraduate students trying to na...,Degree Map AI includes a React front-end and a...,"We first started by working on the scraper, fr...",One challenge we faced was figuring out the be...,We're proud to have successfully connected the...,"We learned to set up OpenAI's ChatGPT, scrape ...","Given the limited amount of time, we were only...","c++, javascript, openai, pyflask, python, react",['https://github.com/april2546/HackRice2024'],['https://github.com/april2546/HackRice2024'],The HackRice2024-DegreeMapAI project is an inn...
6348,HackRice 14 2024,Money Mania,https://devpost.com/software/money-mania,False,Our inspiration for this project was this book...,This game teaches people about personal financ...,We built this in Python using PyGame. We split...,None of us have ever participa

In [2]:
# Only these text columns should be merged
text_cols = [
    'inspiration', 
    'what_it_does', 
    'how_we_built_it', 
    'challenges', 
    'accomplishments', 
    'what_we_learned',
    'what_is_next',
    'tech_stack'
]

# Combine selected columns into one text column
df['all_text'] = df[text_cols].astype(str).apply(lambda x: ' '.join(x), axis=1)

# Clean up text (remove 'nan', brackets, double spaces)
df['all_text'] = (
    df['all_text']
    .str.replace(r"\bnan\b", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [4]:
df_ml = df[['is_winner', 'github_summary', 'all_text']]
df_ml

,is_winner,github_summary,all_text
0,False,"The GitHub repository, ""CalHacksproject,"" is a...",With covid ending and people finding new music...
1,False,"This repository, named `calhacks10.0`, serves ...",Understanding and expressing emotions can be a...
2,False,The Mood-Theremin repository details a CalHack...,"The theremin, an electronic musical instrument..."
3,False,b.a.s.h. (Bay Area Safer Housing) is a CalHack...,The inspiration initially stems from one membe...
4,False,This analysis summarizes the GitHub repository...,"As college students, we both agreed that fun t..."
...,...,...,...
6347,False,The HackRice2024-DegreeMapAI project is an inn...,As current undergraduate students trying to na...
6348,False,"The Hackathon24 project is a ""Personal Finance...",Our inspiration for this project was this book...
6349,False,"The GitHub repository ""Ai_Audio_Analysis"" is d...",The inspiration behind this project came from ...
6350,False,"The GitHub repository ""HackRice14_RecoveryIO,""...",RestoreIO was created in response to the deman...


In [5]:
# Drop rows with missing values (optional)
df_ml = df_ml.dropna(subset=['github_summary', 'all_text'])
df_ml['text'] = df_ml['github_summary']
df_ml['label'] = df_ml['is_winner'].astype(int)

In [6]:
# Sample 1000 True and 1000 False examples randomly
df_true = df_ml[df_ml['is_winner'] == True].sample(n=1000, random_state=42)
df_false = df_ml[df_ml['is_winner'] == False].sample(n=1000, random_state=42)
df_balanced = pd.concat([df_true, df_false]).sample(frac=1, random_state=42).reset_index(drop=True)
df_balanced['is_winner'].value_counts()

is_winner
False    1000
True     1000
Name: count, dtype: int64

In [7]:
df_balanced

,is_winner,github_summary,all_text,text,label
0,False,"The GitHub repository, `repo-sit`, hosts a cro...","With the spread of COVID-19, remote work has b...","The GitHub repository, `repo-sit`, hosts a cro...",0
1,True,This analysis provides a comprehensive overvie...,Introducing Med-Memory: your fast-track soluti...,This analysis provides a comprehensive overvie...,1
2,False,"The GitHub repository, named GTX, presents as ...",Our group members all had different passions a...,"The GitHub repository, named GTX, presents as ...",0
3,True,"This repository, named MedChain_HackPSU, detai...",Medical Health Organization is not adequately ...,"This repository, named MedChain_HackPSU, detai...",1
4,False,The `wanderlust-htn` repository hosts Wanderlu...,"Recently, 3 members of the Wanderlust team had...",The `wanderlust-htn` repository hosts Wanderlu...,0
...,...,...,...,...,...
1995,False,The `luminder` repository contains a dating ap...,"Inspired by VSinder, we combined elements of L...",The `luminder` repository contains a dating ap...,0
1996,False,"The GitHub repository ""QuickTeam"" presents its...",Finding team is a difficult process where dive...,"The GitHub repository ""QuickTeam"" presents its...",0
1997,True,The `3dReal` GitHub repository appears to be a...,"Metaverse, vision pro, spatial video. It’s no ...",The `3dReal` GitHub repository appears to be a...,1
1998,False,This analysis provides a comprehensive overvie...,"As college students, we bear a lot more respon...",This analysis provides a comprehensive overvie...,0


In [9]:
from dotenv import load_dotenv
import os

# Load environment variables from .env file
load_dotenv()

# Access them using os.getenv()
api_key = os.getenv("api")

In [12]:
from google import genai

client = genai.Client(api_key = api_key)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Explain how AI works in a few words"
)
print(response.text)

AI learns patterns from data to make predictions or decisions.


In [19]:
def gemini_responses(df):
    i = -1
    results = []

    for _, row in df.iterrows():
        i +=1
        prompt = f"""
Act as a hackathon judge. Review the Devpost submission and GitHub repository details provided. 
Based on the project’s innovation, completeness, impact, and overall quality, determine whether 
the project qualifies as a winning or non-winning entry.

Return:
"1" → if the project is a winning project  
"0" → if the project is a non-winning project  
Do not return anything else besides "0" or "1".
Devpost submission details: {row['all_text']}  
GitHub repository details: {row['text']}

"""

        # 🔹 Call your model (replace this with the actual API call)
        response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents= prompt
)

        # 🔹 Clean the output: only keep "0" or "1"
        if "1" in response.text:
            result = "1"
        else:
            result = "0"

        results.append(result)
        print(i)

    # Add results as new column in the DataFrame
    df["model_result"] = results

    return df

In [20]:
df_gemini = gemini_responses(df_balanced)
df_gemini

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19


ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'The model is overloaded. Please try again later.', 'status': 'UNAVAILABLE'}}